## Putting it together!

Now we have all of the pieces, we can put them together into a single model.

Lets build a transit model and try it out!

In [ ]:
from taurex.model import TransmissionModel
tm = TransmissionModel(planet=planet,
                       temperature_profile=iso_t,
                       chemistry=chemistry,
                       star=star,
                        pressure_profile=press)

At this point our atmosphere has profiles but no physics! We can add this by including some contributions. Lets add in Absorption:

In [ ]:
from taurex.contributions import AbsorptionContribution
tm.add_contribution(AbsorptionContribution())

Great! No all we have to do is build it:

In [ ]:
tm.build()

``build`` only needs to be done once if none of the models have changed. If you have replaced them then ``build`` needs to be run again.

Now we can actually run our model! The inital run may take a while as data is loaded in:

In [ ]:
wngrid, rprs, tau, _ = tm.model()

Running it again is much faster!

In [ ]:
wngrid, rprs, tau, _ = tm.model()

The output spectral grid is in *wavenumbers* ($cm^{-1}$). We can convert it into wavelength like so:

In [ ]:
wlgrid = 10000/wngrid[::-1]
rprs = rprs[::-1]

Lets plot the results!

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(wlgrid, rprs)
plt.xlabel("Wavelength (um)")
plt.ylabel("$(R_p/R_s)^2$")
plt.xscale("log")

Of course we would like to add some other radiative effects. Lets put in CIA and see what happens:

In [ ]:
from taurex.contributions import CIAContribution
tm.add_contribution(CIAContribution(cia_pairs=['H2-H2','H2-He']))

tm.build()

In [ ]:
wngrid, rprs, tau, _ = tm.model()

wlgrid = 10000/wngrid[::-1]
rprs = rprs[::-1]

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(wlgrid, rprs)
plt.xlabel("Wavelength (um)")
plt.ylabel("$(R_p/R_s)^2$")
plt.xscale("log")

Its subtle but there is a difference, we can see something more dramatic with Rayleigh:

In [ ]:
from taurex.contributions import RayleighContribution
tm.add_contribution(RayleighContribution())

tm.build()

In [ ]:
wngrid, rprs, tau, _ = tm.model()

wlgrid = 10000/wngrid[::-1]
rprs = rprs[::-1]

plt.figure()
plt.plot(wlgrid, rprs)
plt.xlabel("Wavelength (um)")
plt.ylabel("$(R_p/R_s)^2$")
plt.xscale("log")

Ahh yes the distinctive slope!

Lets try using the same objects with Emission:

In [ ]:
from taurex.model import EmissionModel

em = EmissionModel(
    planet=planet,
    temperature_profile=iso_t,
    chemistry=chemistry,
    star=star,
    pressure_profile=press,
)

em.add_contribution(AbsorptionContribution())
em.add_contribution(CIAContribution(cia_pairs=['H2-H2','H2-He']))
em.add_contribution(RayleighContribution())

em.build()

In [ ]:
wngrid, fpfs, tau, _ = em.model()

wlgrid = 10000/wngrid[::-1]
fpfs = fpfs[::-1]

plt.figure()
plt.plot(wlgrid, fpfs)
plt.xlabel("Wavelength (um)")
plt.ylabel("$(F_p/F_s)^2$")
plt.xscale("log")

Ahhh of course, an isothermal will lead to a blackbody spectrum! Okay lets replace the temperature profile with something less constant:

In [ ]:
from taurex.model import EmissionModel
from taurex.temperature import Guillot2010

em = EmissionModel(
    planet=planet,
    temperature_profile=Guillot2010(T_irr=1500),
    chemistry=chemistry,
    star=star,
    pressure_profile=press,
)

em.add_contribution(AbsorptionContribution())
em.add_contribution(CIAContribution(cia_pairs=['H2-H2','H2-He']))
em.add_contribution(RayleighContribution())

em.build()

In [ ]:
wngrid, fpfs, tau, _ = em.model()

wlgrid = 10000/wngrid[::-1]
fpfs = fpfs[::-1]

plt.figure()
plt.plot(wlgrid, fpfs)
plt.xlabel("Wavelength (um)")
plt.ylabel("$(F_p/F_s)^2$")
plt.xscale("log")

# Clouds

TauREx also contains some scattering contributions that simulate the effect of clouds on a spectrum. Some of the *vanilla* cloud models in TauREx include:

- ``FlatMie``
   - A flat absorption across the spectrum with a defined cloud deck
- ``SimpleClouds``
   - An infinitely absorbing cloud deck up to a certain height
- ``LeeMie``
   - A Mie approximation according to [Lee at al 2013](https://iopscience.iop.org/article/10.1088/0004-637X/778/2/97)


We can play around with these clouds and see how they behave. Lets take our current transit model and add some clouds.

## Infinite cloud deck

Lets first try adding in an infinite cloud deck. We can define the top of the cloud deck via ``clouds_pressure``

In [ ]:
from taurex.contributions import SimpleCloudsContribution

clouds = SimpleCloudsContribution(clouds_pressure=1e3)
tm.add_contribution(clouds)
tm.build()

In [ ]:



wngrid, rprs, tau, _ = tm.model()

wlgrid = 10000/wngrid[::-1]
rprs = rprs[::-1]

plt.figure()
plt.plot(wlgrid, rprs)
plt.xlabel("Wavelength (um)")
plt.ylabel("$(R_p/R_s)^2$")
plt.xscale("log")

Interestingly this has the effect of *flattening the spectrum* we can play around with the cloud deck height by modifying the ``cloudsPressure`` variable

In [ ]:
clouds.cloudsPressure = 1e0

wngrid, rprs, tau, _ = tm.model()

wlgrid = 10000/wngrid[::-1]
rprs = rprs[::-1]

plt.figure()
plt.title(f"Cloud deck pressure at {clouds.cloudsPressure} Pa")
plt.plot(wlgrid, rprs)
plt.xlabel("Wavelength (um)")
plt.ylabel("$(R_p/R_s)^2$")
plt.xscale("log")

The infinitely absorbing cloud deck essentially flattens and mutes the spectrum. It has a similar effect of raising the percieved radius of the planet. This make sense as the *true radius* of the planet from a lightcurve would be indistingushable from a thick cloud that absorbs at all wavelengths.

## Flat Cloud

Flat clouds work similarly to the infinite deck but allow us control of how much absorption and where the cloud deck starts as well. The absorption is equal on all wavelengths so we can consider it as *grey*.

Our parameters are:
- ``mix_ratio`` which defines how much absoprtion.
- ``flat_bottomP`` which controls where the cloud deck starts (from BOA)
- ``flat_topP`` which controls where the cloud deck ends. (from BOA)

Setting ``bottomP=-1`` and ``mix_ratio=np.inf`` will be equivalent to the infinite cloud deck.

Before we move on lets define a function to *decloud* the atmosphere, this make it easy for us to replace clouds without recreating the model.




In [ ]:
def decloud(tm):
  tm.contribution_list = [c for c in tm.contribution_list if isinstance(
      c,
      (AbsorptionContribution, CIAContribution, RayleighContribution))]

In [ ]:
decloud(tm)

Now lets add in the flat cloud. We can start it at the bottom of the atmosphere and stop at around $10^3~Pa$

In [ ]:
from taurex.contributions import FlatMieContribution

decloud(tm)

clouds = FlatMieContribution(
    flat_mix_ratio=1e-31,
    flat_bottomP=-1,
    flat_topP=1e3
)

tm.add_contribution(clouds)
tm.build()

wngrid, rprs, tau, _ = tm.model()

wlgrid = 10000/wngrid[::-1]
rprs = rprs[::-1]

plt.figure()
plt.plot(wlgrid, rprs)
plt.xlabel("Wavelength (um)")
plt.ylabel("$(R_p/R_s)^2$")
plt.xscale("log")

The behaviour is a little more interesting as its muting the rayleigh contribution a lot more but not as much as the infinite model.

We can zero the opacity to simulate cloudless atmosphere and increase it significantly to simulate thick clouds (Like the infinite one)

In [ ]:
clouds.mieTopPressure = 1e2
clouds.mieBottomPressure = -1

# No clouds
clouds.mieMixing = 0

wngrid, rprs, tau, _ = tm.model()

wlgrid = 10000/wngrid[::-1]
cloudless_rprs = rprs[::-1]

# Thick clouds
clouds.mieMixing = 1e10

wngrid, rprs, tau, _ = tm.model()

wlgrid = 10000/wngrid[::-1]
thick_rprs = rprs[::-1]

# 'Mid' clouds
clouds.mieMixing = 1e-31

wngrid, rprs, tau, _ = tm.model()

wlgrid = 10000/wngrid[::-1]
partial_rprs = rprs[::-1]


plt.figure()
plt.plot(wlgrid, cloudless_rprs, label="Cloudless")
plt.plot(wlgrid, partial_rprs, label="Partial-Clouds")
plt.plot(wlgrid, thick_rprs, label="Thick Clouds")
plt.xlabel("Wavelength (um)")
plt.ylabel("$(R_p/R_s)^2$")
plt.legend()
plt.xscale("log")

Play around with the top/bottom and mixing values and see how the clouds change the behaviour of the spectrum!

## LeeMie

``LeeMie`` clouds are a Mie scattering approximation that accounts for particle radius, extinction and density. They require a little more care in accurately modelling but a more "physically motivated" approximation to cloud behaviour.

As before lets decloud our model:

In [ ]:
decloud(tm)

First lets store a cloudless version of the atmosphere for comparison.

In [ ]:
# lets take a cloudless model first
tm.build()
wngrid, rprs, tau, _ = tm.model()

wlgrid = 10000/wngrid[::-1]
cloudless_rprs = rprs[::-1]


Now lets add LeeMie to the model:

In [ ]:
from taurex.contributions import LeeMieContribution
decloud(tm)
clouds = LeeMieContribution(
)

tm.add_contribution(clouds)
tm.build()

Now we model and plot:

In [ ]:
# Particle radius
clouds.mieRadius = 0.01
# Mie extinction
clouds.mieQ = 40
# Particle mixing in particles/m3
clouds.mieMixing = 1e14
# Top Deck pressure
clouds.mieTopPressure = 1e1
# bottom deck pressure
clouds.mieBottomPressure = -1


wngrid, rprs, tau, _ = tm.model()

wlgrid = 10000/wngrid[::-1]
lee_rprs = rprs[::-1]

plt.figure()
plt.plot(wlgrid, cloudless_rprs, label="Cloudless")
plt.plot(wlgrid, lee_rprs, label="LeeMie")
plt.xlabel("Wavelength (um)")
plt.ylabel("$(R_p/R_s)^2$")
plt.legend()
plt.xscale("log")
plt.savefig("simulation.svg")


This model provides a wavelength dependant cloud model, as you can see there is a flattening in the optical wavelengths but this reduces as we move to the IR. This makes sense as there is a dependancy with particle radius and scattering. Try modifying the particle radius to ``0.1`` and see what happens

Lee Mie is a decent approximation simulate clouds that are not *grey*.


# Inspecting the model

We can inspect various aspects of the model we've build by accesing its properties. For example, lets see the temperature profile:

In [ ]:
plt.figure()
plt.plot(em.temperatureProfile, em.pressureProfile)
plt.xlabel("Temperature (K)")
plt.ylabel("Pressure (Pa)")
# Flip the Y axis
plt.gca().invert_yaxis()
plt.yscale("log")


Or even the chemical profile

In [ ]:
plt.figure()

for x,gasname in enumerate(em.chemistry.gases):

    plt.plot(em.chemistry.mixProfile[x],em.pressureProfile/1e5,label=gasname)

plt.gca().invert_yaxis()
plt.yscale("log")
plt.xscale("log")
plt.xlabel("VMR")
plt.ylabel("Pressure (bar)")
plt.legend()
plt.show()

Or properties like the density, altitude, mean molecular weight

In [ ]:
from taurex.constants import AMU
fig, (ax1,ax2,ax3) = plt.subplots(1,3,figsize=(15,5))

ax1.plot(em.densityProfile,em.pressureProfile/1e5)
ax1.invert_yaxis()
ax1.set_xlabel('Density [kg/m3]')
ax1.set_ylabel('Pressure [bar]')
ax1.set_yscale("log")

ax2.plot(em.altitudeProfile,em.pressureProfile/1e5)
ax2.invert_yaxis()
ax2.set_xlabel('Altitude [m]')
ax2.set_ylabel('Pressure [bar]')
ax2.set_yscale("log")
ax3.plot(em.chemistry.mu_profile/AMU,em.pressureProfile/1e5)
ax3.invert_yaxis()
ax3.set_xlabel('Mean Molecular Weight [amu]')
ax3.set_ylabel('Pressure [bar]')
ax3.set_yscale("log")

# Input file

Input files that are loaded into the ``taurex`` program can also be used to load in models.

For example, the transmission model we have defined has an equivalent cloudless version as an input file

🚀**TIP** We are generating an input file on the spot by doing this.

In [ ]:
from pathlib import Path

input_par = f"""[Global]
xsec_path = {XSEC_DIR}
cia_path = {CIA_DIR}


[Chemistry]
chemistry_type = taurex
fill_gases = H2,He
ratio = 0.1741

    [[H2O]]
    gas_type=constant
    mix_ratio = 1e-3

    [[CH4]]
    gas_type=constant
    mix_ratio = 1e-4

    [[NH3]]
    gas_type=constant
    mix_ratio = 1e-4

    [[CO2]]
    gas_type=constant
    mix_ratio = 1e-4

[Temperature]
profile_type = isothermal
T = 2000

[Pressure]
profile_type = Simple
atm_min_pressure = 1e-5
atm_max_pressure = 1e5
nlayers = 100

[Planet]
planet_type = Simple
planet_mass = 0.74
planet_radius = 1.38

[Star]
star_type = blackbody
temperature = 6117
radius = 1.16

[Model]
model_type = transmission

    [[Absorption]]
    [[CIA]]
    cia_pairs = H2-H2,H2-He
    [[Rayleigh]]
"""

Path("input.par").write_text(input_par)
print(Path("input.par").read_text())

In [ ]:
! taurex -i input.par -o output.hdf5

In [ ]:
!taurex-plot --help

In [ ]:
!convert output_spectrum_contrib_forward.pdf output_spectrum_contrib_forward.png

We can load in this par file like so:

In [ ]:
from taurex.parameter import ParameterParser

pp = ParameterParser()

pp.read("input.par")
pp.setup_globals()

tm = pp.generate_appropriate_model()

In [ ]:
tm.build()

wngrid, rprs, tau, _ = tm.model()

wlgrid = 10000/wngrid[::-1]
rprs = rprs[::-1]

plt.figure()
plt.plot(wlgrid, rprs)
plt.xscale('log')
plt.xlabel('Wavelength (nm)')
plt.ylabel('$(R_p/R_s)^2$')
plt.show()

🚀 **TIP**: You can find out how define a parfile from the [taurex3 documentation](https://taurex3.readthedocs.io/en/latest/user/taurex/index.html)

# Updating the model

It is possible to update a models parameters after it has been built. Each model can use the square bracket notation ``[]`` to access parameters and modify them after the fact. This is one of TauREx-3's key features.

For example we can pump up the temperature to 6000 K:

In [ ]:
tm["T"] = 2000.0

In [ ]:
wngrid, rprs, tau, _ = tm.model()

wlgrid = 10000/wngrid[::-1]
rprs = rprs[::-1]

plt.figure()
plt.plot(wlgrid, rprs)
plt.xscale('log')
plt.xlabel('Wavelength (nm)')
plt.ylabel('$(R_p/R_s)^2$')
plt.show()

And remove water entirely:

In [ ]:
tm["H2O"] = 1e-3


wngrid, rprs, tau, _ = tm.model()

wlgrid = 10000/wngrid[::-1]
rprs = rprs[::-1]

plt.figure()
plt.plot(wlgrid, rprs)
plt.xscale('log')
plt.xlabel('Wavelength (nm)')
plt.ylabel('$(R_p/R_s)^2$')
plt.show()

These brackets can also be used to see what the current variable is:

In [ ]:
print(f"Temperature is at {tm['T']}")
print(f"VMR of CO2 is {tm['CO2']}")


To see the full parameters available to modify we can use the ``fittingParameters`` attribute.

In [ ]:
list(tm.fittingParameters.keys())

When we learn about retrievals we will understand why this feature of TauREx 3 is so important.

# Contributions to Spectra

We will also go through the process of seeing how each part of atmosphere and its processes contribute tot he final spectrum and flux of the planet.



## Depth contribution

We want to assess how each layer in the atmosphere contributes to the spectrum. This information is provided by the optical depth ($\tau$) from the modelling.

In [ ]:
wngrid, rprs, tau, _ = tm.model()

wlgrid = 10000/wngrid[::-1] # cm-1 to um
rprs = rprs[::-1]
tau = tau[...,::-1]

plt.figure()
plt.imshow(tau,aspect="auto")
plt.colorbar()
plt.show()


We dont have too much information about this without proper context so lets build a nicer way of plotting this:

In [ ]:
import numpy as np

def plot_optical_depth(ax, tau, pressure, wavelengths,
                       wl_ticks=6,
                       pressure_unit="Pa"):
  import numpy as np
  from matplotlib import ticker
  # Plot the optical depth
  im = ax.pcolormesh(wavelengths, pressure,tau)
  ax.set_xlabel("Wavelength (um)")
  ax.set_ylabel(f"Pressure ({pressure_unit})")
  ax.set_yscale("log")
  ax.set_xscale("log")
  start_wl = np.log10(wavelengths.min())
  end_wl = np.log10(wavelengths.max())

  ticks_wl = np.logspace(start_wl, end_wl, wl_ticks)

  ax.xaxis.set_major_locator(ticker.FixedLocator(ticks_wl))
  ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.1f"))
  ax.invert_yaxis()
  # Add minor ticks for y axis
  locmin = ticker.LogLocator(base=10.0,subs=(0.2,0.4,0.6,0.8),numticks=12)
  ax.yaxis.set_minor_locator(locmin)
  ax.yaxis.set_minor_formatter(ticker.NullFormatter())



  return im



In [ ]:
fig, ax = plt.subplots()
im = plot_optical_depth(ax, tau, tm.pressureProfile/1e5, wlgrid,pressure_unit="bar")
fig.colorbar(im)
plt.show()

A much nicer plot! However it would also be good to see the average contribution at depth. We can compute this by taking the average along wavelength:

In [ ]:
def depth_contribution(tau):
    return np.average(tau, axis=1)

In [ ]:
depth_contribution(tau)

Lets put them together and see what the depth-contribution looks like.

In [ ]:
# Lets create a grid specification
grid = plt.GridSpec(1, 4, wspace=0.4, hspace=0.3)
fig = plt.figure()
ax1 = fig.add_subplot(grid[0, :3])
ax2 = fig.add_subplot(grid[0, 3], sharey=ax1)

im = plot_optical_depth(ax1, tau, tm.pressureProfile/1e5, wlgrid, pressure_unit="bar")
ax2.plot(depth_contribution(tau), tm.pressureProfile/1e5)
ax2.yaxis.tick_right()
ax2.set_xlabel("Contribution")
plt.show()


Now we have a really good idea on how each layer influences the spectrum!

## Radiative Transfer contributions

We have a good idea of how deep we can see into the atmosphere, let us also examine how each part of the ``Contributions`` influences the shape of the spectrum!

When we model a spectrum, we can ask TauREx to model each contribution in isolation! This can be achieved with the ``model_full_contrib`` function. Lets try it out!

In [ ]:
wngrid, contribs = tm.model_full_contrib()
contribs

This is a fairly large dataset. Lets examine it carefully:

In [ ]:
contribs.keys()

We have each of the "contributions" to the spectrum listed. Lets examine the first one:

In [ ]:
len(contribs["Absorption"])

These are each of the molecules in the spectrum! Lets see one of them!

In [ ]:
absorption = contribs["Absorption"]
absorption[0]

The first is the ``label``, the second it the resultant spectrum (in this case $(R_p/R_s)^2$) if it was the only contributor. Lastly is the optical depth. We can plot these individually!

In [ ]:
label, abs_rprs, abs_tau, _ = absorption[0]

plt.figure()
plt.plot(wlgrid, rprs, label="Full spectrum")
plt.plot(wlgrid, abs_rprs[::-1], label=label)
plt.xlabel("Wavelength (um)")
plt.ylabel("$(R_p/R_s)^2$")
plt.xscale("log")
plt.legend()
plt.show()

Of course we removed water from last time. Lets update the model and try again!

In [ ]:
# Update our model parameters
tm["T"] = 1000.0
tm["H2O"] = 1e-4

# Remodel the full spectrum
wngrid, rprs, tau, _ = tm.model()

wlgrid = 10000/wngrid[::-1]
rprs = rprs[::-1]

# Model the contributions
wngrid, contribs = tm.model_full_contrib()

absorption = contribs["Absorption"]

# Extract water
label, abs_rprs, abs_tau, _ = absorption[0]

# Plot
plt.figure()
plt.plot(wlgrid, rprs, label="Full spectrum")
plt.plot(wlgrid, abs_rprs[::-1], label=label, alpha=0.6)
plt.xlabel("Wavelength (um)")
plt.ylabel("$(R_p/R_s)^2$")
plt.xscale("log")
plt.legend()
plt.show()

Very nice! Lets examing a different one, ``Rayleigh``

In [ ]:
rayleigh = contribs["Rayleigh"]
label, ray_rprs, ray_tau, _ = rayleigh[4]

plt.figure()
plt.plot(wlgrid, rprs, label="Full spectrum")
plt.plot(wlgrid, ray_rprs[::-1], label=label, alpha=0.6)
plt.xlabel("Wavelength (um)")
plt.ylabel("$(R_p/R_s)^2$")
plt.xscale("log")
plt.legend()

Nice! So we can see that the slope came from $H_2$ rayleigh scattering. And of-course we can examine at which depth is it mostly contributing:

In [ ]:
# Lets create a grid specification
grid = plt.GridSpec(1, 4, wspace=0.4, hspace=0.3)
fig = plt.figure()
ax1 = fig.add_subplot(grid[0, :3])
ax2 = fig.add_subplot(grid[0, 3], sharey=ax1)
fig.suptitle(f"Rayleigh Scattering by {label}")
im = plot_optical_depth(ax1, ray_tau, tm.pressureProfile/1e5, wlgrid, pressure_unit="bar")
ax2.plot(depth_contribution(ray_tau), tm.pressureProfile/1e5)
ax2.yaxis.tick_right()
ax2.set_xlabel("Contribution")
plt.show()

# Exercises

1. Try modelling clouds and examining the optical depth contribution.
2. How do each cloud look in their spectral contributions
3. Add clouds to emission and see what happens, why does this occur?
  - Which clouds do/don't work with emission. Why is this the case?
4. Add new molecules and see how they behave in their spectral contributions.
  - You can use the ``wget`` script at the top and add get molecule links from the [Exomol](https://www.exomol.com) to download new cross-sections.